In [11]:
# import model configs
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
from pathlib import Path
import pandas as pd
import numpy as np


from src.paths import *
from src.nano_maker.container.configs import *
from src.nano_maker.skeleton import Skeleton
from src.nano_maker.nanomaker import NanoMaker
from src.nano_maker.modules.data_handling.radial_dataset import RadialDataset
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint


from tqdm import tqdm

In [12]:
# Loss Curve Plotting

In [13]:
RADIAL_SEQUENCES = pd.read_pickle(DATABASE / "radial_seq_df.pkl")
RADIAL_SEQUENCES.set_index("PDB_ID", inplace=True)

In [14]:
MOLECULAR_FINGERPRINTS = pd.read_pickle(DATABASE / "scaffold_molfp_df.pkl")
MOLECULAR_FINGERPRINTS.set_index("smiles_str", inplace=True)

In [15]:
test_pointer_path = DATABASE / "skeleton_test_pointers_dict.parquet"

In [16]:
# Analyze Protein Pocket Biochemistry as a proxy for potential zero-shot capability or model accuracy
# We expect LogP of a given drug to correlate with low hydrophobicity of a given protein pocket
# polar drug = hydrophilic protein pocket
# ok won't be able to run this until i finish naanobot I3
# 20 drug structures of varying LogPs -> 10 protein pockets generated for each
# per drug structure's LogP -> get mean protein pocket polarity
# plot

model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_cfg=naanobot_config,
                  radial_cfg=radial_config)

In [17]:
drug_i_want_to_deliver = "C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|"
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|
Scaffold: c1ccc(-c2ccc(N3C[C@@H]4C[C@H]3C[NH2+]4)nn2)cc1
Drug Likeness Rules Passed: 4 / 4


In [29]:
@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=120, sampling_temperature=0.2, verbose=True):
    model.eval()

    map4_enc = map4_enc.to(next(model.parameters()).device)

    # generate starter context -> we want to isolate naanobot here. otherwise wouldn't be able to tell if biochemical changes came from 3D coordinates
    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    chain_data = ""
    position_log = []

    # precompute normalized AA matrix once
    aa_ids = [aa_id for aa_id in model.naano_module.nAAno_vectors.keys() if aa_id != "VOID"]
    n_v_tensor = torch.tensor(
        [model.naano_module.nAAno_vectors[aa_id] for aa_id in aa_ids],
        dtype=torch.float32
    )
    aa_norm = F.normalize(n_v_tensor, dim=-1)  # [n_aa, features]

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(
            coord_context, bioch_context, coordinate
        ).unsqueeze(0).to(map4_enc.device)

        output, _ = model.forward(naano_X, map4_enc)
        vector = output[0].detach().float()

        # cosine similarities — consistent with loss and approx_id
        pred_norm = F.normalize(vector.unsqueeze(0), dim=-1)
        similarities = (pred_norm @ aa_norm.T).squeeze(0)
        ranked = sorted(zip(aa_ids, similarities.tolist()), key=lambda x: -x[1])
        top5_names = [aa for aa, _ in ranked[:5]]
        greedy = top5_names[0]

        # sample
        sampled = model.naano_module.approx_id(output[0], sampling_temperature=sampling_temperature)
        diverged = sampled != greedy

        if verbose:
            top5_str = "  ".join(top5_names)
            diverge_str = f"  → sampled {sampled}" if diverged else ""
            print(f"[{i:2d}] r={coordinate[0]:.1f}Å  top5: {top5_str}{diverge_str}")

        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(sampled)]
        coord_context = coord_context[1:] + [coordinate]
        chain_data += sampled

        position_log.append({
            "position": i,
            "coordinate": coordinate,
            "greedy": greedy,
            "sampled": sampled,
            "top5": top5_names,
            "diverged": diverged
        })

    if verbose:
        divergences = sum(1 for p in position_log if p["diverged"])
        print(f"\nchain:      {chain_data}")
        print(f"diverged:   {divergences}/{len(position_log)} ({100*divergences/len(position_log):.0f}%)")

    return chain_data

In [30]:
map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()

# wide range of coordinates
sph_coordinates =[
    [18.42,  1.832,  0.847],
    [16.91, -2.441,  2.103],
    [15.73,  0.291,  1.234],
    [14.88, -1.057,  0.612],
    [13.55,  2.718,  2.441],
    [12.34, -0.883,  1.876],
    [11.92,  1.204,  0.394],
    [10.67, -2.093,  2.718],
    [ 9.44,  0.751,  1.047],
    [ 8.83, -1.571,  0.628],
    [ 7.62,  2.094,  2.356],
    [ 6.51, -0.314,  1.309],
    [ 5.38,  1.047,  0.785],
    [ 4.22, -2.618,  2.094],
    [ 3.14,  0.524,  1.571],
]

diagnose_generation(model=model._NAAnoBotPrototype, map4_enc=map4_enc, sph_coordinates=sph_coordinates)

[ 0] r=18.4Å  top5: N  R  Q  H  D  → sampled T
[ 1] r=16.9Å  top5: R  K  Y  H  N  → sampled N
[ 2] r=15.7Å  top5: D  E  N  S  Q  → sampled E
[ 3] r=14.9Å  top5: Y  R  K  W  F  → sampled C
[ 4] r=13.6Å  top5: D  S  N  H  T  → sampled N
[ 5] r=12.3Å  top5: D  S  N  E  T  → sampled V
[ 6] r=11.9Å  top5: T  V  S  N  I  → sampled M
[ 7] r=10.7Å  top5: D  E  T  S  N  → sampled Q
[ 8] r=9.4Å  top5: T  S  N  Q  A  → sampled I
[ 9] r=8.8Å  top5: S  G  N  T  D  → sampled D
[10] r=7.6Å  top5: T  H  S  D  N  → sampled L
[11] r=6.5Å  top5: S  N  T  Q  H
[12] r=5.4Å  top5: S  N  G  D  T  → sampled H
[13] r=4.2Å  top5: S  D  N  T  Q  → sampled Q
[14] r=3.1Å  top5: S  G  T  A  N  → sampled L

chain:      TNECNVMQIDLSHQL
diverged:   14/15 (93%)


'TNECNVMQIDLSHQL'